In [1]:
import os
import re
import json
import math
from pathlib import Path
from typing import List, Dict, Tuple

import fitz  # PyMuPDF
import pandas as pd
from tqdm.auto import tqdm

In [2]:

# CONFIG

STUDENT_ID = "st125002"
LAST_DIGIT = int(STUDENT_ID[-1])

# Assignment mapping rule
CHAPTER_NUM = 10 if LAST_DIGIT == 0 else LAST_DIGIT

# Update this path to your downloaded Chapter 9 PDF
CHAPTER_PDF_PATH = "data/chapter_2.pdf"

# Output folder
OUTPUT_DIR = Path("outputs/task1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_TEXT_PATH = OUTPUT_DIR / f"chapter_{CHAPTER_NUM}_raw.txt"
CLEAN_TEXT_PATH = OUTPUT_DIR / f"chapter_{CHAPTER_NUM}_clean.txt"
PARAGRAPHS_PATH = OUTPUT_DIR / f"chapter_{CHAPTER_NUM}_paragraphs.json"
QA_PATH = OUTPUT_DIR / f"chapter_{CHAPTER_NUM}_qa_pairs.json"

print("Student ID:", STUDENT_ID)
print("Assigned Chapter:", CHAPTER_NUM)
print("PDF Path:", CHAPTER_PDF_PATH)
print("Output Dir:", OUTPUT_DIR)

Student ID: st125002
Assigned Chapter: 2
PDF Path: data/chapter_2.pdf
Output Dir: outputs/task1


### Helper functions for chapter mapping and validation

In [3]:
def get_assigned_chapter(student_id: str) -> int:
    """
    Return assigned chapter based on the last digit of the student ID.
    Rule from assignment:
    - if last digit = 0 => chapter 10
    - otherwise chapter = last digit
    """
    last_digit = int(student_id[-1])
    return 10 if last_digit == 0 else last_digit


def validate_assignment_setup(student_id: str, expected_chapter: int) -> None:
    assigned = get_assigned_chapter(student_id)
    assert assigned == expected_chapter, (
        f"Mismatch: student ID {student_id} maps to chapter {assigned}, "
        f"but config says chapter {expected_chapter}"
    )
    print(f"Validated: {student_id} -> Chapter {assigned}")

### PDF text extraction

In [4]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract raw text from a PDF using PyMuPDF.
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    doc = fitz.open(pdf_path)
    all_pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        all_pages.append(text)

    full_text = "\n".join(all_pages)
    return full_text

### Text cleaning utilities

In [5]:
def remove_page_numbers(text: str) -> str:
    """
    Remove isolated page numbers that often appear in PDF extraction.
    """
    lines = text.splitlines()
    cleaned = []

    for line in lines:
        stripped = line.strip()
        if re.fullmatch(r"\d+", stripped):
            continue
        cleaned.append(line)

    return "\n".join(cleaned)


def remove_extra_whitespace(text: str) -> str:
    """
    Normalize whitespace while preserving paragraph breaks.
    """
    # normalize spaces/tabs inside lines
    text = re.sub(r"[ \t]+", " ", text)

    # normalize too many blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def fix_hyphenated_linebreaks(text: str) -> str:
    """
    Join words broken by PDF line-wrap hyphenation.
    Example:
        'informa-\ntion' -> 'information'
    """
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return text


def join_broken_lines(text: str) -> str:
    """
    Merge lines that are likely part of the same paragraph.
    Keep true paragraph breaks.
    """
    # replace single newlines between non-empty lines with a space
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    return text


def clean_chapter_text(raw_text: str) -> str:
    """
    Main cleaning pipeline.
    """
    text = raw_text

    # remove weird unicode if needed
    text = text.replace("\x0c", " ")
    text = text.replace("\u00a0", " ")

    text = fix_hyphenated_linebreaks(text)
    text = remove_page_numbers(text)

    # preserve paragraph structure first:
    # turn multiple line breaks into double breaks before line joining
    text = re.sub(r"\n\s*\n", "\n\n", text)
    text = join_broken_lines(text)
    text = remove_extra_whitespace(text)

    return text

### Paragraph segmentation

In [6]:
def split_into_paragraphs(clean_text: str, min_chars: int = 80) -> List[Dict]:
    """
    Split clean text into paragraph-like units.
    Filters out very short noisy fragments.
    """
    raw_paragraphs = re.split(r"\n\s*\n", clean_text)
    paragraphs = []

    for idx, para in enumerate(raw_paragraphs):
        para = para.strip()
        if len(para) < min_chars:
            continue

        paragraphs.append({
            "paragraph_id": len(paragraphs),
            "text": para,
            "char_length": len(para),
            "word_count": len(para.split())
        })

    return paragraphs

### Run extraction and cleaning

In [7]:
validate_assignment_setup(STUDENT_ID, CHAPTER_NUM)

raw_text = extract_text_from_pdf(CHAPTER_PDF_PATH)
clean_text = clean_chapter_text(raw_text)
paragraphs = split_into_paragraphs(clean_text)

RAW_TEXT_PATH.write_text(raw_text, encoding="utf-8")
CLEAN_TEXT_PATH.write_text(clean_text, encoding="utf-8")

with open(PARAGRAPHS_PATH, "w", encoding="utf-8") as f:
    json.dump(paragraphs, f, indent=2, ensure_ascii=False)

print("Raw text saved to:", RAW_TEXT_PATH)
print("Clean text saved to:", CLEAN_TEXT_PATH)
print("Paragraphs saved to:", PARAGRAPHS_PATH)
print("Total paragraphs:", len(paragraphs))
print("\nSample paragraph:\n")
print(paragraphs[0]["text"][:1000] if paragraphs else "No paragraphs found.")

Validated: st125002 -> Chapter 2
Raw text saved to: outputs/task1/chapter_2_raw.txt
Clean text saved to: outputs/task1/chapter_2_clean.txt
Paragraphs saved to: outputs/task1/chapter_2_paragraphs.json
Total paragraphs: 36

Sample paragraph:

Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER Words and Tokens User: I need some help, that much seems certain. ELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP User: Perhaps I could learn to get along with my mother. ELIZA: TELL ME MORE ABOUT YOUR FAMILY User: My mother takes care of me. ELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU User: My father. ELIZA: YOUR FATHER User: You are like my father in some ways. Weizenbaum (1966) The dialogue above is from ELIZA, an early natural language processing system ELIZA that could carry on a limited conversation with a user by imitating the responses of a Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is 

### Quick inspection helpers

In [8]:
def preview_paragraphs(paragraphs: List[Dict], n: int = 5) -> None:
    """
    Print a few paragraphs for manual inspection.
    """
    for item in paragraphs[:n]:
        
        print(f"Paragraph ID: {item['paragraph_id']}")
        print(f"Words: {item['word_count']}")
        print(item["text"][:1200])
        print()


preview_paragraphs(paragraphs, n=3)

Paragraph ID: 0
Words: 533
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER Words and Tokens User: I need some help, that much seems certain. ELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP User: Perhaps I could learn to get along with my mother. ELIZA: TELL ME MORE ABOUT YOUR FAMILY User: My mother takes care of me. ELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU User: My father. ELIZA: YOUR FATHER User: You are like my father in some ways. Weizenbaum (1966) The dialogue above is from ELIZA, an early natural language processing system ELIZA that could carry on a limited conversation with a user by imitating the responses of a Rogerian psychotherapist (Weizenbaum, 1966). ELIZA is a surprisingly simple program that uses pattern matching on words to recognize phrases like “I need X” and change the words into suitable outputs like “What would it mean to you if you got X?”. ELIZA’s mimicry of h

### Task 1 Part B — Generate QA pairs

In [9]:
import json
import re
from typing import List, Dict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

### Model config

In [10]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Model:", MODEL_NAME)

Using device: cuda
Model: Qwen/Qwen2.5-3B-Instruct


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.


### Build text-generation pipeline

In [12]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

### Prompt builder for QA generation

In [13]:
def build_qa_generation_prompt(group_text: str, chapter_num: int, n_pairs: int = 4) -> str:
    prompt = f"""
Creating a question-answer dataset for a Retrieval-Augmented Generation assignment.

Chapter number: {chapter_num}

Generate EXACTLY {n_pairs} question-answer pairs from the SOURCE TEXT below.

Rules:
1. Use ONLY the source text.
2. Do NOT use outside knowledge.
3. Do NOT guess.
4. Each question must cover a DIFFERENT concept.
5. Questions must be specific and factual.
6. Answers must be short, accurate.
7. evidence_excerpt must be a short supporting excerpt in ONE LINE only.
8. Return ONLY valid JSON.
9. Do NOT use markdown code fences.
10. Do NOT include any explanation before or after the JSON.

Output format:
[
  {{
    "question": "one-line question",
    "answer": "one-line answer",
    "evidence_excerpt": "one-line evidence"
  }}
]

SOURCE TEXT:
\"\"\"
{group_text[:7000]}
\"\"\"
"""
    return prompt.strip()

### Generate raw response from Qwen

In [14]:
def generate_with_qwen(
    prompt: str,
    max_new_tokens: int = 450,
    temperature: float = 0.3
) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are a careful academic assistant that returns valid JSON only."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    outputs = generator(
        text_input,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        return_full_text=False
    )

    return outputs[0]["generated_text"].strip()

### JSON parser for model output

In [15]:
import json
import re

def clean_json_text(text: str) -> str:
    """
    Clean common JSON formatting issues from LLM output.
    """
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # normalize curly quotes to straight quotes
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")

    # remove null bytes / problematic control chars except newline/tab
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", text)

    return text.strip()


def extract_json_array(text: str):
    """
    Extract and parse the first JSON array from model output.
    More tolerant than direct json.loads.
    """
    text = clean_json_text(text)

    # Try direct parse first
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try extracting the first [...] block
    match = re.search(r"\[\s*{.*}\s*\]", text, re.DOTALL)
    if not match:
        raise ValueError("Could not find JSON array in model output.")

    candidate = match.group(0)

    # Remove bad control chars again
    candidate = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", candidate)

    # Try direct parse
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Escape raw newlines that appear inside quoted JSON strings
    repaired = []
    in_string = False
    escape = False

    for ch in candidate:
        if ch == '"' and not escape:
            in_string = not in_string
        if ch == "\n" and in_string:
            repaired.append("\\n")
            continue
        if ch == "\r" and in_string:
            repaired.append("\\r")
            continue
        if ch == "\t" and in_string:
            repaired.append("\\t")
            continue

        repaired.append(ch)

        if ch == "\\" and not escape:
            escape = True
        else:
            escape = False

    repaired_text = "".join(repaired)

    # Remove trailing commas before ] or }
    repaired_text = re.sub(r",\s*([}\]])", r"\1", repaired_text)

    return json.loads(repaired_text)

### QA generation function

In [16]:
def generate_qa_pairs_from_group_qwen(
    group_text: str,
    chapter_num: int,
    n_pairs: int = 2
) -> List[Dict]:
    """
    Generate QA pairs from one paragraph group using Qwen.
    """
    prompt = build_qa_generation_prompt(
        group_text=group_text,
        chapter_num=chapter_num,
        n_pairs=n_pairs
    )

    raw_output = generate_with_qwen(
        prompt=prompt,
        max_new_tokens=500,
        temperature=0.2
    )

    qa_pairs = extract_json_array(raw_output)

    # minimal validation
    cleaned = []
    for item in qa_pairs:
        if not isinstance(item, dict):
            continue

        question = str(item.get("question", "")).strip()
        answer = str(item.get("answer", "")).strip()
        evidence = str(item.get("evidence_excerpt", "")).strip()

        if question and answer:
            cleaned.append({
                "question": question,
                "answer": answer,
                "evidence_excerpt": evidence
            })

    return cleaned

### Build paragraph groups

In [17]:
def group_paragraphs_for_qa(paragraphs: List[Dict], batch_size: int = 4) -> List[Dict]:
    """
    Group paragraphs so the model has enough context for QA generation.
    """
    groups = []

    for i in range(0, len(paragraphs), batch_size):
        batch = paragraphs[i:i + batch_size]
        combined_text = "\n\n".join(
            [f"[Paragraph {p['paragraph_id']}]\n{p['text']}" for p in batch]
        )

        groups.append({
            "group_id": len(groups),
            "paragraph_ids": [p["paragraph_id"] for p in batch],
            "text": combined_text
        })

    return groups

In [18]:
def generate_qa_pairs_from_group_qwen(
    group_text: str,
    chapter_num: int,
    n_pairs: int = 4,
    max_retries: int = 3
):
    """
    Generate QA pairs from one paragraph group using Qwen, with retries.
    """
    prompt = build_qa_generation_prompt(
        group_text=group_text,
        chapter_num=chapter_num,
        n_pairs=n_pairs
    )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            raw_output = generate_with_qwen(
                prompt=prompt,
                max_new_tokens=500,
                temperature=0.3
            )

            qa_pairs = extract_json_array(raw_output)

            cleaned = []
            for item in qa_pairs:
                if not isinstance(item, dict):
                    continue

                question = str(item.get("question", "")).strip()
                answer = str(item.get("answer", "")).strip()
                evidence = str(item.get("evidence_excerpt", "")).strip()

                if question and answer:
                    cleaned.append({
                        "question": question,
                        "answer": answer,
                        "evidence_excerpt": evidence
                    })

            if len(cleaned) > 0:
                return cleaned

        except Exception as e:
            last_error = e
            print(f"Retry {attempt}/{max_retries} failed: {e}")

    raise ValueError(f"Failed after {max_retries} retries. Last error: {last_error}")

### Generate candidate QA pairs from all groups

In [19]:
groups = group_paragraphs_for_qa(paragraphs, batch_size=4)
print("Total groups:", len(groups))

Total groups: 9


In [20]:
candidate_qa_pairs = []

for round_idx in range(2):
    print(f"\n=== Generation Round {round_idx + 1} ===")

    for group in tqdm(groups, desc=f"Round {round_idx + 1}"):
        try:
            qa_items = generate_qa_pairs_from_group_qwen(
                group_text=group["text"],
                chapter_num=CHAPTER_NUM,
                n_pairs=4,
                max_retries=3
            )

            for item in qa_items:
                candidate_qa_pairs.append({
                    "group_id": group["group_id"],
                    "paragraph_ids": group["paragraph_ids"],
                    "question": item["question"],
                    "answer": item["answer"],
                    "evidence_excerpt": item["evidence_excerpt"]
                })

        except Exception as e:
            print(f"Error in group {group['group_id']}: {e}")

print("Generated candidate QA pairs:", len(candidate_qa_pairs))


=== Generation Round 1 ===


Round 1:   0%|          | 0/9 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/

Retry 1/3 failed: Expecting ',' delimiter: line 10 column 167 (char 659)


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Retry 1/3 failed: Invalid \escape: line 20 column 149 (char 1291)


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retry 1/3 failed: Expecting ',' delimiter: line 10 column 60 (char 349)


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retry 2/3 failed: Expecting ',' delimiter: line 15 column 60 (char 603)
Retry 3/3 failed: Expecting ',' delimiter: line 15 column 112 (char 1240)
Error in group 8: Failed after 3 retries. Last error: Expecting ',' delimiter: line 15 column 112 (char 1240)
Generated candidate QA pairs: 60


### Remove duplicates

In [21]:
def normalize_text_for_compare(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def deduplicate_qa_pairs(qa_pairs: List[Dict]) -> List[Dict]:
    seen = set()
    unique = []

    for item in qa_pairs:
        key = normalize_text_for_compare(item["question"])
        if key not in seen:
            seen.add(key)
            unique.append(item)

    return unique

In [22]:
candidate_qa_pairs = deduplicate_qa_pairs(candidate_qa_pairs)
print("Unique QA pairs:", len(candidate_qa_pairs))

Unique QA pairs: 54


### Filter weak outputs

In [23]:
def filter_weak_qa_pairs(qa_pairs: List[Dict]) -> List[Dict]:
    filtered = []

    bad_prefixes = [
        "what does this text say",
        "what is mentioned",
        "what is discussed"
    ]

    for item in qa_pairs:
        q = item["question"].strip()
        a = item["answer"].strip()

        if len(q.split()) < 4:
            continue
        if len(a.split()) < 5:
            continue
        if any(q.lower().startswith(prefix) for prefix in bad_prefixes):
            continue

        filtered.append(item)

    return filtered

In [24]:
candidate_qa_pairs = filter_weak_qa_pairs(candidate_qa_pairs)
print("After filtering:", len(candidate_qa_pairs))

After filtering: 29


### Select final 20 QA pairs

In [25]:
def select_final_qa_pairs(qa_pairs: List[Dict], target_count: int = 20) -> List[Dict]:
    if len(qa_pairs) < target_count:
        raise ValueError(f"Need at least {target_count} QA pairs, but only got {len(qa_pairs)}.")

    final = qa_pairs[:target_count]

    for i, item in enumerate(final, start=1):
        item["qa_id"] = i

    return final

In [26]:
final_qa_pairs = select_final_qa_pairs(candidate_qa_pairs, target_count=20)

df_qa = pd.DataFrame(final_qa_pairs)
df_qa[["qa_id", "question", "answer", "evidence_excerpt", "paragraph_ids"]]

,qa_id,question,answer,evidence_excerpt,paragraph_ids
0,1,What is the difference between word types and ...,Word types are the number of distinct words in...,Word types word type are the number of distinc...,"[0, 1, 2, 3]"
1,2,What is the difference between orthographic wo...,Orthographic words are words based on our Engl...,"For example, while orthographically I'm is one...","[0, 1, 2, 3]"
2,3,What does an inﬂectional morpheme do?,"It tends to play a syntactic role, such as mar...",Inﬂectional morphemes tend to play a syntactic...,"[4, 5, 6, 7]"
3,4,What is the difference between analytic and sy...,"Analytic languages have one morpheme per word,...",The first dimension is the number of morphemes...,"[4, 5, 6, 7]"
4,5,What is UTF-32 encoding?,It's a 4-byte representation that makes files ...,With this 4-byte representation the word hello...,"[8, 9, 10, 11]"
5,6,What does UTF-8 do for characters above 127?,"It uses fewer bytes for them, encoding them as...","For a given code point in the From-To range, t...","[8, 9, 10, 11]"
6,7,What is the advantage of UTF-8 over UTF-32?,"It's relatively efficient, using fewer bytes f...",UTF-8 has a number of advantages. It's relativ...,"[8, 9, 10, 11]"
7,8,What does BPE do in the BPE encoder?,It tokenizes a test sentence using merges lear...,"Once we've learned our vocabulary, the BPE enc...","[12, 13, 14, 15]"
8,9,What happens to unseen combinations like revis...,Morphemes like the re- prefix are created.,But the merges also created knowledge of morph...,"[12, 13, 14, 15]"
9,10,How does BPE handle Unicode input?,It treats each byte of UTF-8-encoded text as i...,We normally run BPE on the individual bytes of...,"[12, 13, 14, 15]"


### Save dataset

In [27]:
QA_PATH = OUTPUT_DIR / f"chapter_{CHAPTER_NUM}_qa_pairs.json"

with open(QA_PATH, "w", encoding="utf-8") as f:
    json.dump(final_qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved QA dataset to:", QA_PATH)

Saved QA dataset to: outputs/task1/chapter_2_qa_pairs.json


### Export simplified evaluation dataset

In [28]:
def export_ground_truth_qa(qa_pairs: List[Dict], path):
    simplified = [
        {
            "question": item["question"],
            "ground_truth_answer": item["answer"]
        }
        for item in qa_pairs
    ]

    with open(path, "w", encoding="utf-8") as f:
        json.dump(simplified, f, indent=2, ensure_ascii=False)

In [29]:
GROUND_TRUTH_QA_PATH = OUTPUT_DIR / f"chapter_{CHAPTER_NUM}_ground_truth_qa.json"
export_ground_truth_qa(final_qa_pairs, GROUND_TRUTH_QA_PATH)

print("Saved evaluation QA file to:", GROUND_TRUTH_QA_PATH)

Saved evaluation QA file to: outputs/task1/chapter_2_ground_truth_qa.json


# Task 2

In [30]:
import json
import re
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

### Paths and config

In [31]:

# CONFIG


STUDENT_ID = "st125002"
CHAPTER_NUM = 2

BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "outputs"
TASK1_DIR = OUTPUT_DIR / "task1"
TASK2_DIR = OUTPUT_DIR / "task2"
TASK2_DIR.mkdir(parents=True, exist_ok=True)

PARAGRAPHS_PATH = TASK1_DIR / f"chapter_{CHAPTER_NUM}_paragraphs.json"
GROUND_TRUTH_QA_PATH = TASK1_DIR / f"chapter_{CHAPTER_NUM}_ground_truth_qa.json"

NAIVE_RESULTS_PATH = TASK2_DIR / "naive_rag_outputs.json"
CONTEXTUAL_RESULTS_PATH = TASK2_DIR / "contextual_rag_outputs.json"
FINAL_RESPONSE_PATH = TASK2_DIR / f"response-{STUDENT_ID}-chapter-{CHAPTER_NUM}.json"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

TOP_K = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Generator model:", GEN_MODEL_NAME)

Device: cuda
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Generator model: Qwen/Qwen2.5-3B-Instruct


### Load Task 1 outputs

In [32]:
with open(PARAGRAPHS_PATH, "r", encoding="utf-8") as f:
    paragraphs = json.load(f)

with open(GROUND_TRUTH_QA_PATH, "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)

print("Paragraphs:", len(paragraphs))
print("QA pairs:", len(qa_dataset))

pd.DataFrame(qa_dataset).head()

Paragraphs: 36
QA pairs: 20


,question,ground_truth_answer
0,What is the difference between word types and ...,Word types are the number of distinct words in...
1,What is the difference between orthographic wo...,Orthographic words are words based on our Engl...
2,What does an inﬂectional morpheme do?,"It tends to play a syntactic role, such as mar..."
3,What is the difference between analytic and sy...,"Analytic languages have one morpheme per word,..."
4,What is UTF-32 encoding?,It's a 4-byte representation that makes files ...


### Part A — Prepare chunks for retrieval

### Build naive chunks

In [33]:
def build_naive_chunks(paragraphs: List[Dict]) -> List[Dict]:
    """
    Use Task 1 paragraph units as the base retrieval chunks.
    """
    chunks = []

    for p in paragraphs:
        chunks.append({
            "chunk_id": p["paragraph_id"],
            "source_paragraph_id": p["paragraph_id"],
            "text": p["text"]
        })

    return chunks

In [34]:
naive_chunks = build_naive_chunks(paragraphs)
print("Naive chunks:", len(naive_chunks))
print(naive_chunks[0]["text"][:500])

Naive chunks: 36
Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER Words and Tokens User: I need some help, that much seems certain. ELIZA: WHAT WOULD IT MEAN TO YOU IF YOU GOT SOME HELP User: Perhaps I could learn to get along with my mother. ELIZA: TELL ME MORE ABOUT YOUR FAMILY User: My mother takes care of me. ELIZA: WHO ELSE IN YOU FAMILY TAKES CARE OF YOU User: My father. ELIZA: YOUR FATHER User: You are like my fathe


### Part B — Load embedding model

### Load sentence embedding model

In [35]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


### Encode chunks

In [36]:
def encode_texts(texts: List[str], model: SentenceTransformer, batch_size: int = 32) -> np.ndarray:
    """
    Encode texts into dense vectors.
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings.astype("float32")

In [37]:
naive_chunk_texts = [c["text"] for c in naive_chunks]
naive_chunk_embeddings = encode_texts(naive_chunk_texts, embedding_model)

print("Chunk embedding shape:", naive_chunk_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Chunk embedding shape: (36, 384)


### Part C — Build FAISS index

### Build vector index

In [38]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """
    Build an inner-product FAISS index.
    Since embeddings are normalized, inner product behaves like cosine similarity.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

In [39]:
naive_index = build_faiss_index(naive_chunk_embeddings)
print("FAISS index built.")

FAISS index built.


### Retrieval function

In [40]:
def retrieve_top_k(
    query: str,
    chunks: List[Dict],
    index: faiss.Index,
    embedding_model: SentenceTransformer,
    top_k: int = 3
) -> List[Dict]:
    """
    Retrieve top-k most relevant chunks for a question.
    """
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        chunk = chunks[int(idx)]
        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "source_paragraph_id": chunk["source_paragraph_id"],
            "text": chunk["text"]
        })

    return results

In [41]:
sample_results = retrieve_top_k(
    query=qa_dataset[0]["question"],
    chunks=naive_chunks,
    index=naive_index,
    embedding_model=embedding_model,
    top_k=TOP_K
)

sample_results

[{'rank': 1,
  'score': 0.5394628047943115,
  'chunk_id': 2,
  'source_paragraph_id': 2,
  'text': '2.1 • WORDS Corpus Types = |V| Instances = N Shakespeare 31 thousand 884 thousand Brown corpus 38 thousand 1 million Switchboard telephone conversations 20 thousand 2.4 million COCA 2 million 440 million Google n-grams 13 million 1 trillion Figure 2.1 Rough numbers of wordform types and instances for some English language corpora. The largest, the Google n-grams corpus, contains 13 million types, but this count only includes types appearing 40 or more times, so the true number would be much larger. The distinctions get even harder to make once we start to think about other languages. For example the writing systems of languages like Chinese, Japanese, and Thai simply don’t have orthographic words at all! That is, they don’t use spaces to mark potential word-boundaries. In Chinese, for example, words are composed of characters (called hanzi in Chinese). Each character generally represents

### Part D — Load Qwen generator

### Load tokenizer and model

In [42]:
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

generator_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Generator model loaded.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Generator model loaded.


### Build generation pipeline

In [43]:
text_generator = pipeline(
    "text-generation",
    model=generator_model,
    tokenizer=tokenizer
)

### Part E — Answer generation for Naive RAG

### Build Naive RAG prompt

In [44]:
def build_naive_rag_prompt(question: str, retrieved_chunks: List[Dict]) -> str:
    """
    Prompt for answer generation using retrieved chunks only.
    """
    context_blocks = []
    for item in retrieved_chunks:
        context_blocks.append(
            f"[Chunk {item['chunk_id']}]\n{item['text']}"
        )

    context_text = "\n\n".join(context_blocks)

    prompt = f"""
You are answering a question using only the retrieved chapter context.

Rules:
1. Use ONLY the context below.
2. Do NOT use outside knowledge.
3. If the answer is not clearly supported by the context, say so briefly.
4. Write a concise answer in 1-3 sentences.

Question:
{question}

Retrieved Context:
{context_text}

Answer:
"""
    return prompt.strip()

In [45]:
def generate_answer_with_qwen(
    prompt: str,
    max_new_tokens: int = 180,
    temperature: float = 0.2
) -> str:
    """
    Generate a grounded answer from Qwen.
    """
    messages = [
        {"role": "system", "content": "You are a careful academic assistant."},
        {"role": "user", "content": prompt}
    ]

    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    outputs = text_generator(
        text_input,
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=0.9,
        return_full_text=False
    )

    answer = outputs[0]["generated_text"].strip()
    return answer

In [46]:
def run_naive_rag_pipeline(
    qa_dataset: List[Dict],
    chunks: List[Dict],
    index: faiss.Index,
    embedding_model: SentenceTransformer,
    top_k: int = 3
) -> List[Dict]:
    """
    Run naive RAG on all questions.
    """
    outputs = []

    for item in tqdm(qa_dataset, desc="Running Naive RAG"):
        question = item["question"]
        ground_truth = item["ground_truth_answer"]

        retrieved = retrieve_top_k(
            query=question,
            chunks=chunks,
            index=index,
            embedding_model=embedding_model,
            top_k=top_k
        )

        prompt = build_naive_rag_prompt(question, retrieved)
        answer = generate_answer_with_qwen(prompt)

        outputs.append({
            "question": question,
            "ground_truth_answer": ground_truth,
            "naive_rag_answer": answer,
            "retrieved_chunks": retrieved
        })

    return outputs

In [47]:
naive_outputs = run_naive_rag_pipeline(
    qa_dataset=qa_dataset,
    chunks=naive_chunks,
    index=naive_index,
    embedding_model=embedding_model,
    top_k=TOP_K
)

Running Naive RAG:   0%|          | 0/20 [00:00<?, ?it/s]

Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

In [48]:
with open(NAIVE_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(naive_outputs, f, indent=2, ensure_ascii=False)

print("Saved:", NAIVE_RESULTS_PATH)

Saved: outputs/task2/naive_rag_outputs.json


### Contextual Retrieval

### Build contextualization prompt

In [51]:
def build_chunk_context_prompt(chunk_text: str, full_document_text: str, title: str) -> str:
    """
    Prompt for generating a short contextual prefix for a chunk.
    """
    prompt = f"""
You are enriching a chunk for contextual retrieval.

Document title: {title}

Below is the document excerpt and the target chunk.

Your job:
Write 1-2 short sentences explaining what this chunk discusses in relation to the full document.
Be specific and concise.
Return only the context sentence(s), no bullet points, no markdown.

Document excerpt:
{full_document_text[:4000]}

Target chunk:
{chunk_text}

Context:
"""
    return prompt.strip()

### Generate contextual prefix

In [52]:
def generate_chunk_context(
    chunk_text: str,
    full_document_text: str,
    title: str = "Chapter 2"
) -> str:
    """
    Generate contextual prefix for one chunk.
    """
    prompt = build_chunk_context_prompt(
        chunk_text=chunk_text,
        full_document_text=full_document_text,
        title=title
    )

    context = generate_answer_with_qwen(
        prompt=prompt,
        max_new_tokens=100,
        temperature=0.2
    )

    return context.strip()

### Build contextual chunks

In [53]:
full_document_text = "\n\n".join([p["text"] for p in paragraphs])

In [54]:
def build_contextual_chunks(
    naive_chunks: List[Dict],
    full_document_text: str,
    title: str = "Chapter 2"
) -> List[Dict]:
    """
    Create contextual retrieval chunks by prepending generated context.
    """
    contextual_chunks = []

    for chunk in tqdm(naive_chunks, desc="Building contextual chunks"):
        context_prefix = generate_chunk_context(
            chunk_text=chunk["text"],
            full_document_text=full_document_text,
            title=title
        )

        enriched_text = f"{context_prefix}\n\n{chunk['text']}"

        contextual_chunks.append({
            "chunk_id": chunk["chunk_id"],
            "source_paragraph_id": chunk["source_paragraph_id"],
            "context_prefix": context_prefix,
            "original_text": chunk["text"],
            "text": enriched_text
        })

    return contextual_chunks

In [55]:
contextual_chunks = build_contextual_chunks(
    naive_chunks=naive_chunks,
    full_document_text=full_document_text,
    title="Chapter 2"
)

Building contextual chunks:   0%|          | 0/36 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

### Save contextual chunks

In [56]:
CONTEXTUAL_CHUNKS_PATH = TASK2_DIR / "contextual_chunks.json"

with open(CONTEXTUAL_CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(contextual_chunks, f, indent=2, ensure_ascii=False)

print("Saved:", CONTEXTUAL_CHUNKS_PATH)

Saved: outputs/task2/contextual_chunks.json


### Embed contextual chunks and build index

In [57]:
contextual_chunk_texts = [c["text"] for c in contextual_chunks]
contextual_chunk_embeddings = encode_texts(contextual_chunk_texts, embedding_model)
contextual_index = build_faiss_index(contextual_chunk_embeddings)

print("Contextual embeddings shape:", contextual_chunk_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Contextual embeddings shape: (36, 384)


### Build Contextual RAG prompt

In [58]:
def build_contextual_rag_prompt(question: str, retrieved_chunks: List[Dict]) -> str:
    """
    Same answer-generation logic, but retrieved chunks come from contextual retrieval.
    """
    context_blocks = []
    for item in retrieved_chunks:
        context_blocks.append(
            f"[Chunk {item['chunk_id']}]\n{item['text']}"
        )

    context_text = "\n\n".join(context_blocks)

    prompt = f"""
You are answering a question using only the retrieved chapter context.

Rules:
1. Use ONLY the context below.
2. Do NOT use outside knowledge.
3. If the answer is not clearly supported by the context, say so briefly.
4. Write a concise answer in 1-3 sentences.

Question:
{question}

Retrieved Context:
{context_text}

Answer:
"""
    return prompt.strip()

### Run Contextual Retrieval pipeline

In [59]:
def run_contextual_rag_pipeline(
    qa_dataset: List[Dict],
    chunks: List[Dict],
    index: faiss.Index,
    embedding_model: SentenceTransformer,
    top_k: int = 3
) -> List[Dict]:
    """
    Run contextual retrieval pipeline on all questions.
    """
    outputs = []

    for item in tqdm(qa_dataset, desc="Running Contextual Retrieval"):
        question = item["question"]
        ground_truth = item["ground_truth_answer"]

        retrieved = retrieve_top_k(
            query=question,
            chunks=chunks,
            index=index,
            embedding_model=embedding_model,
            top_k=top_k
        )

        prompt = build_contextual_rag_prompt(question, retrieved)
        answer = generate_answer_with_qwen(prompt)

        outputs.append({
            "question": question,
            "ground_truth_answer": ground_truth,
            "contextual_retrieval_answer": answer,
            "retrieved_chunks": retrieved
        })

    return outputs

In [60]:
contextual_outputs = run_contextual_rag_pipeline(
    qa_dataset=qa_dataset,
    chunks=contextual_chunks,
    index=contextual_index,
    embedding_model=embedding_model,
    top_k=TOP_K
)

Running Contextual Retrieval:   0%|          | 0/20 [00:00<?, ?it/s]

Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

### Save Contextual Retrieval outputs

In [61]:
with open(CONTEXTUAL_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(contextual_outputs, f, indent=2, ensure_ascii=False)

print("Saved:", CONTEXTUAL_RESULTS_PATH)

Saved: outputs/task2/contextual_rag_outputs.json


### Part G — ROUGE evaluation

### ROUGE utilities

In [62]:
def compute_average_rouge(
    references: List[str],
    predictions: List[str]
) -> Dict[str, float]:
    """
    Compute average ROUGE-1, ROUGE-2, ROUGE-L F1 scores.
    """
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    scores = {
        "rouge1": [],
        "rouge2": [],
        "rougeL": []
    }

    for ref, pred in zip(references, predictions):
        result = scorer.score(ref, pred)
        scores["rouge1"].append(result["rouge1"].fmeasure)
        scores["rouge2"].append(result["rouge2"].fmeasure)
        scores["rougeL"].append(result["rougeL"].fmeasure)

    avg_scores = {
        metric: float(np.mean(values))
        for metric, values in scores.items()
    }

    return avg_scores

### Evaluate Naive vs Contextual

In [63]:
naive_references = [item["ground_truth_answer"] for item in naive_outputs]
naive_predictions = [item["naive_rag_answer"] for item in naive_outputs]

context_references = [item["ground_truth_answer"] for item in contextual_outputs]
context_predictions = [item["contextual_retrieval_answer"] for item in contextual_outputs]

naive_rouge = compute_average_rouge(naive_references, naive_predictions)
contextual_rouge = compute_average_rouge(context_references, context_predictions)

naive_rouge, contextual_rouge

({'rouge1': 0.19778947171378172,
  'rouge2': 0.08453002840878507,
  'rougeL': 0.15697993678313277},
 {'rouge1': 0.22477773081930078,
  'rouge2': 0.11450471322154479,
  'rougeL': 0.18321844166448212})

In [64]:
evaluation_table = pd.DataFrame([
    {
        "Method": "Naive RAG",
        "ROUGE-1": round(naive_rouge["rouge1"], 4),
        "ROUGE-2": round(naive_rouge["rouge2"], 4),
        "ROUGE-L": round(naive_rouge["rougeL"], 4),
    },
    {
        "Method": "Contextual Retrieval",
        "ROUGE-1": round(contextual_rouge["rouge1"], 4),
        "ROUGE-2": round(contextual_rouge["rouge2"], 4),
        "ROUGE-L": round(contextual_rouge["rougeL"], 4),
    }
])

evaluation_table

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Naive RAG,0.1978,0.0845,0.1570
1,Contextual Retrieval,0.2248,0.1145,0.1832


In [65]:
def merge_final_outputs(
    naive_outputs: List[Dict],
    contextual_outputs: List[Dict]
) -> List[Dict]:
    """
    Merge naive and contextual answers into the required submission format.
    Assumes same question order.
    """
    final_data = []

    for naive_item, context_item in zip(naive_outputs, contextual_outputs):
        final_data.append({
            "question": naive_item["question"],
            "ground_truth_answer": naive_item["ground_truth_answer"],
            "naive_rag_answer": naive_item["naive_rag_answer"],
            "contextual_retrieval_answer": context_item["contextual_retrieval_answer"]
        })

    return final_data

In [66]:
final_response_data = merge_final_outputs(naive_outputs, contextual_outputs)

with open(FINAL_RESPONSE_PATH, "w", encoding="utf-8") as f:
    json.dump(final_response_data, f, indent=2, ensure_ascii=False)

print("Saved final response file to:", FINAL_RESPONSE_PATH)

Saved final response file to: outputs/task2/response-st125002-chapter-2.json


In [67]:
pd.DataFrame(final_response_data).head()

,question,ground_truth_answer,naive_rag_answer,contextual_retrieval_answer
0,What is the difference between word types and ...,Word types are the number of distinct words in...,Word types refer to unique lexical items in a ...,Word types refer to unique lexical items in a ...
1,What is the difference between orthographic wo...,Orthographic words are words based on our Engl...,Orthographic words refer to words that are cle...,Orthographic words refer to units of meaning i...
2,What does an inﬂectional morpheme do?,"It tends to play a syntactic role, such as mar...",An inﬂectional morpheme modifies the syntactic...,An inﬂectional morpheme plays a syntactic role...
3,What is the difference between analytic and sy...,"Analytic languages have one morpheme per word,...",The context does not directly address the diff...,"Analytic languages, like Farsi and English, ty..."
4,What is UTF-32 encoding?,It's a 4-byte representation that makes files ...,UTF-32 encoding uses 4 bytes per character to ...,UTF-32 encoding uses 4 bytes (32 bits) to repr...
